In [2]:

#Imports
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.data import load_matches, load_elo_baseline
from src.elo import prepare_matches, run_elo_updates, update_elo
from src.features import get_stats, update_team_history, get_h2h_stat, update_h2h
from collections import deque,defaultdict

import pandas as pd

sys.path.append(os.path.abspath(".."))



In [3]:
# Load data
df = load_matches() #Load dataframe containing all historical matches
country_elo = load_elo_baseline() #Generate a dictionary with every country having their elo from 12/13/2025

# Prepare and update
recent_matches = prepare_matches(df, start_date="2025-12-13", end_date="2026-06-11") #We need these games since the current elo is only accurate up til 12/13/2025
country_elo = run_elo_updates(recent_matches, country_elo)

#Top 20
top20 = sorted(country_elo.items(), key=lambda x: x[1], reverse=True)[:20]
print("Top 20 teams going into the WC based off ELO")
for rank, (team, rating) in enumerate(top20, 1):
    print(f"{rank}. {team}: {rating:.0f}")

Top 20 teams going into the WC based off ELO
1. Spain: 2163
2. Argentina: 2113
3. France: 2081
4. England: 2020
5. Brazil: 1990
6. Portugal: 1984
7. Colombia: 1975
8. Netherlands: 1962
9. Ecuador: 1937
10. Croatia: 1931
11. Germany: 1926
12. Norway: 1912
13. Japan: 1906
14. Turkey: 1902
15. Switzerland: 1893
16. Uruguay: 1892
17. Morocco: 1876
18. Mexico: 1868
19. Belgium: 1866
20. Denmark: 1859


ELO caculations turn out essentially the same to the elo rating found on https://www.eloratings.net/2026_World_Cup, 

In [8]:
current_elos = defaultdict(lambda: 1500)
team_history = defaultdict(lambda: deque(maxlen=10))
h2h_history = defaultdict(lambda: deque(maxlen=5))
training_rows = []


df_sorted = prepare_matches(df, df['date'].min(), pd.Timestamp("2026-06-11")).sort_values('date')

for row in df_sorted.itertuples(index=False):
    
    home_elo = current_elos[row.home_team]
    away_elo = current_elos[row.away_team]
    
    h_form, h_gd = get_stats(row.home_team, team_history)
    a_form, a_gd = get_stats(row.away_team, team_history)
    h2h_val = get_h2h_stat(row.home_team, row.away_team, h2h_history)
        
    training_rows.append({
        'date': row.date,
        'home_team': row.home_team,
        'away_team': row.away_team,
        'elo_diff': home_elo-away_elo,
        'home_form': h_form,
        'away_form': a_form,
        'h2h': h2h_val,
        'home_gd': h_gd,
        'away_gd': a_gd,
        'neutral': row.neutral,
        'target': row.result
    })
    
    new_away, new_home = update_elo(
        row.result, 
        row.neutral, 
        row.K_factor, 
        row.home_score, 
        row.away_score, 
        current_elos[row.away_team], 
        current_elos[row.home_team]
    )
    
    current_elos[row.home_team] = new_home
    current_elos[row.away_team] = new_away
    
    update_team_history(row, team_history)
    update_h2h(row, h2h_history)
    
# 3. Finalize
train_df = pd.DataFrame(training_rows)
train_df=train_df.round(2)
train_df.to_csv("/Users/armand_k/Workout app/WC 2026/data/world_cup_training_data.csv", index=False)


/var/folders/f0/0r4c9w2d0vx1d7zk05qxcr4h0000gn/T/ipykernel_28804/2754352929.py:50: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  train_df=train_df.round(2)


File saved successfully!
